# Malay-English Lemmatizer Token Demo

Activate the five comparison systems, enter one sentence, and inspect each system's token-level output.

In [4]:
from pathlib import Path
import gc
import os
import re
import sys

import pandas as pd

# Resolve the repository root when this notebook is opened from the src/ folder.
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "src" else Path.cwd().resolve()
os.chdir(PROJECT_ROOT)
src_dir = PROJECT_ROOT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from lemmatizer_systems import build_system

SYSTEM_KEYS = ["malaya_naive", "sastrawi", "stem_lstm_512", "sailor2", "llama"]
TOKEN_RE = re.compile(r"[A-Za-z]+(?:['-][A-Za-z]+)*|\d+|[^\w\s]", re.UNICODE)

print(f"Project root: {PROJECT_ROOT}")
print(f"Systems: {', '.join(SYSTEM_KEYS)}")

Project root: C:\Y3S1\nlp\assignment\Sailor2_Lemma
Systems: malaya_naive, sastrawi, stem_lstm_512, sailor2, llama


## Activate all systems

This cell constructs every requested system. Large model checkpoints are loaded only when their prediction is run.

In [3]:
systems = {}
activation_errors = {}

for key in SYSTEM_KEYS:
    try:
        systems[key] = build_system(key)
        print(f"Activated: {key}")
    except Exception as exc:
        activation_errors[key] = f"{type(exc).__name__}: {exc}"
        print(f"Could not activate {key}: {activation_errors[key]}")

print(f"Ready: {len(systems)}/{len(SYSTEM_KEYS)} systems")

Activated: malaya_naive
Activated: sastrawi
[stem_lstm_512] loading mesolitica/stem-lstm-512 via malaya.stem.huggingface ...
Activated: stem_lstm_512
Could not activate sailor2: FileNotFoundError: no merged checkpoint at ./trained_models/sailor2_malay_lemmatizer (run the matching train_*_lora.py first)
Could not activate llama: FileNotFoundError: no merged checkpoint at ./trained_models/llama_malay_lemmatizer (run the matching train_*_lora.py first)
Ready: 3/5 systems


## Enter a sentence

In [3]:
sentence = input("Enter a Malay-English sentence: ").strip()
if not sentence:
    raise ValueError("Please enter a non-empty sentence.")

print(f"Input: {sentence}")

Input: Wifi kat rumah aku slow gila, lag teruk time main online.


## Token-level outputs

Each row is one token. `surface` is the input token and `lemma` is that system's output.

In [ ]:
def release_system_resources(system):
    if hasattr(system, "_stemmer"):
        system._stemmer = None
    if hasattr(system, "_model"):
        system._model = None
    if hasattr(system, "_tokenizer"):
        system._tokenizer = None
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


# Baseline systems require target surfaces for identical tokenisation.
# Lemmas are set to the surface because this is an inference demo, not gold evaluation.
surfaces = TOKEN_RE.findall(sentence)
rows = [{
    "sentence": sentence,
    "target": [{"surface": token, "lemma": token} for token in surfaces],
}]

token_rows = []
run_errors = dict(activation_errors)

for key, system in systems.items():
    try:
        predictions = system.predict_batch(rows)
        prediction = predictions[0] if predictions else None
        if prediction is None:
            token_rows.append({
                "System": system.name,
                "Token": "<unparseable>",
                "Surface": "",
                "Lemma": "",
                "Status": "unparseable output",
            })
        else:
            token_number = 0
            for item in prediction:
                if not isinstance(item, dict) or "surface" not in item or "lemma" not in item:
                    continue
                token_number += 1
                surface = str(item["surface"])
                lemma = str(item["lemma"])
                token_rows.append({
                    "System": system.name,
                    "Token": token_number,
                    "Surface": surface,
                    "Lemma": lemma,
                    "Status": "changed" if surface.lower() != lemma.lower() else "unchanged",
                })
    except Exception as exc:
        run_errors[key] = f"{type(exc).__name__}: {exc}"
        token_rows.append({
            "System": key,
            "Token": "<error>",
            "Surface": "",
            "Lemma": "",
            "Status": run_errors[key],
        })
    finally:
        release_system_resources(system)

output_table = pd.DataFrame(token_rows)
display(output_table)

if run_errors:
    print("Systems with activation or inference errors:")
    display(pd.DataFrame([{"System": key, "Reason": reason} for key, reason in run_errors.items()]))

[sailor2_ft] loading ./trained_models/sailor2_malay_lemmatizer ...


Loading weights: 100%|██████████| 387/387 [00:00<00:00, 1781.88it/s]


[sailor2_ft] loaded in 188.4s on cuda:0
